In [1]:
import pandas as pd 
import numpy as np

In [2]:
campaigns = pd.read_csv("Campaign_Raw.csv",encoding='latin1')
Shopify_sales = pd.read_csv("Raw_Shopify_Sales.csv",encoding='latin1')

In [3]:
#data observation and checking nulls in each column
campaigns.info()
campaigns.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10028 entries, 0 to 10027
Data columns (total 26 columns):
 #   Column                                       Non-Null Count  Dtype  
---  ------                                       --------------  -----  
 0   Data Source name                             9399 non-null   object 
 1   Date                                         9395 non-null   object 
 2   Campaign Name                                9392 non-null   object 
 3   Campaign Effective Status                    9383 non-null   object 
 4   Ad Set Name                                  9387 non-null   object 
 5   Ad Name                                      9382 non-null   object 
 6   Country Funnel                               9232 non-null   object 
 7   Geo Location Segment                         9360 non-null   object 
 8   FB Spent Funnel (INR)                        9278 non-null   float64
 9   Amount Spent (INR)                           9358 non-null   float64
 10

Data Source name                                629
Date                                            633
Campaign Name                                   636
Campaign Effective Status                       645
Ad Set Name                                     641
Ad Name                                         646
Country Funnel                                  796
Geo Location Segment                            668
FB Spent Funnel (INR)                           750
Amount Spent (INR)                              670
Clicks (all)                                    666
Impressions                                     668
Page Likes                                      906
Landing Page Views                              669
Link Clicks                                     672
Adds to Cart                                    659
Checkouts Initiated                            1045
Adds of Payment Info                           1078
Purchases                                       673
Purchases Co

In [4]:
#checking duplicates
campaigns.duplicated().sum()

np.int64(310)

In [5]:
# remove duplicates first
campaigns = campaigns.drop_duplicates()

In [6]:
#fixing date column data type
campaigns['Date'] = pd.to_datetime(campaigns['Date'],errors = 'coerce')

In [7]:
#fixed date dat type first because many nulls  in date are due  to incorrect data type


In [8]:
campaigns.isnull().sum() 
# as we can see date had almost 6000 nulls even after correcting data type so we should drop it because date column is necessary for analysis later

Data Source name                                612
Date                                           5984
Campaign Name                                   621
Campaign Effective Status                       620
Ad Set Name                                     614
Ad Name                                         620
Country Funnel                                  765
Geo Location Segment                            635
FB Spent Funnel (INR)                           725
Amount Spent (INR)                              643
Clicks (all)                                    638
Impressions                                     650
Page Likes                                      884
Landing Page Views                              648
Link Clicks                                     654
Adds to Cart                                    631
Checkouts Initiated                            1013
Adds of Payment Info                           1042
Purchases                                       658
Purchases Co

In [9]:
campaigns = campaigns.dropna(subset=['Date'])

In [10]:
campaigns.isnull().sum()

Data Source name                                233
Date                                              0
Campaign Name                                   255
Campaign Effective Status                       241
Ad Set Name                                     239
Ad Name                                         224
Country Funnel                                  298
Geo Location Segment                            226
FB Spent Funnel (INR)                           268
Amount Spent (INR)                              247
Clicks (all)                                    234
Impressions                                     259
Page Likes                                      282
Landing Page Views                              238
Link Clicks                                     274
Adds to Cart                                    244
Checkouts Initiated                             306
Adds of Payment Info                            303
Purchases                                       255
Purchases Co

In [11]:
# now we have  to handle nulls for otehr columns we can not remove them blindly 
# as we have two type column text and numeric columns we will fill numeric ones with 0 and text ones with unknown

In [12]:
numeric_cols = [
    'FB Spent Funnel (INR)', 'Amount Spent (INR)', 'Clicks (all)','Purchases Conversion Value (INR)',
    'Impressions', 'Adds to Cart', 'Purchases','Page Likes','Landing Page Views','Link Clicks','Messaging Conversations Started',
     'Adds to Cart Conversion Value (INR)','Checkouts Initiated Conversion Value (INR)','Adds of Payment Info Conversion Value (INR)',
    'Checkouts Initiated','Adds of Payment Info','Row Count','Website Contacts'
]

campaigns[numeric_cols] = campaigns[numeric_cols].fillna(0)

In [13]:
text_cols = [
    'Campaign Name', 'Ad Name', 'Country Funnel',
    'Geo Location Segment', 'Campaign Effective Status','Data Source name','Ad Set Name'
]

campaigns[text_cols] = campaigns[text_cols].fillna('Unknown')

In [14]:
# now we remove extra spaces and make all string value in lower case so that there is no case error 

In [15]:
for col in text_cols:
    campaigns[col] = campaigns[col].str.strip().str.lower()

In [16]:
# these columns have  some  values 'nan' instead of blank 
col_having_nan = ['Data Source name','Ad Name','Campaign Effective Status','Country Funnel','Geo Location Segment']
for cols in col_having_nan:
    campaigns[cols] = campaigns[cols].replace('nan','unknown')

In [17]:
campaigns['Country Funnel'] = campaigns['Country Funnel'].str.title() # to make country name look as title 

In [18]:
campaigns['Campaign Effective Status'].unique() # no 'nan'

array(['paused', 'unknown', 'active'], dtype=object)

In [19]:
parts = campaigns['Campaign Name'].str.split('|', expand=True) # splitting values into multple column as per assignment 

In [20]:
parts.head(10)

,0,1,2,3,4,5,6
0,unknown,None,None,None,None,None,None
1,growify,11th june,tof,abo,india,lal 3% - 5%,sales
3,growify,11th june,tof,abo,india,lal 3% - 5%,sales
4,growify,11th june,tof,abo,india,lal 3% - 5%,sales
10,growify,16th june,tof,india,asc,cac,sales
12,growify,23rd sept,tof,abo,sales,None,None
17,growify,2nd december,abo,sale,None,None,None
18,growify,2nd december,abo,sale,None,None,None
19,growify,2nd december,abo,sale,None,None,None
20,growify,2nd december,abo,sale,None,None,None


In [21]:
campaigns['brand'] = parts[0]
campaigns['funnel_stage'] = parts[2]
campaigns['region'] = parts[3]

In [22]:
campaigns[['brand', 'funnel_stage', 'region']] = campaigns[['brand', 'funnel_stage', 'region']].fillna('Unknown')

In [23]:
campaigns['brand'] = campaigns['brand'].str.strip().str.lower()
campaigns['funnel_stage'] = campaigns['funnel_stage'].str.strip().str.lower()
campaigns['region'] = campaigns['region'].str.strip().str.lower()

In [24]:
campaigns['brand'] = campaigns['brand'].replace('nan','unknown')
campaigns['funnel_stage'] = campaigns['funnel_stage'].replace('nan','unknown')
campaigns['funnel_stage'] = campaigns['funnel_stage'].replace('none','unknown')
campaigns['region'] = campaigns['region'].replace('nan','unknown')
campaigns['region'] = campaigns['region'].replace('none','unknown')

In [25]:
campaigns['region'].unique()

array(['unknown', 'abo', 'india', 'sale', 'sales', 'open', 'intl', 'cpr',
       'engagement', 'uae', 'aus+can+uk+usa', 'lehenga+saree', 'us+can',
       'us', 'can'], dtype=object)

In [26]:
#now we do same thing for ad_parts  but this data is very bad so we can not get proper insights 

In [27]:
ad_parts  = campaigns['Ad Set Name'].str.split('|', expand=True)

In [28]:
ad_parts.head(10)

,0,1,2,3,4
0,tof,lal 3-5%,sales,womens,None
1,tof,lal 3-5%,sales,womens,None
3,tof,lal 3-5%,sales,womens,None
4,unknown,None,None,None,None
10,tof,asc,cac,india,None
12,market places - tof - sales - growify,None,None,None,None
17,tof,engaged shoppers,sale,women,None
18,tof,engaged shoppers,sale,women,None
19,tof,engaged shoppers,sale,women,None
20,tof,engaged shoppers,sale,women,None


In [29]:
campaigns['channel'] = ad_parts[0]
campaigns['persona']  = ad_parts[1]

In [30]:
campaigns[['channel','persona']] = campaigns[['channel','persona']].fillna('unknown')

In [31]:
campaigns['channel'] = campaigns['channel'].str.strip().str.lower()
campaigns['persona'] = campaigns['persona'].str.strip().str.lower()
campaigns['channel'] = campaigns['channel'].replace('nan','unknown')
campaigns['persona'] = campaigns['persona'].replace('nan','unknown')

In [32]:
campaigns['persona'].unique()

array(['lal 3-5%', 'unknown', 'asc', 'engaged shoppers', 'whtsapp cd',
       'new collection', 'rt > video only', 'apparel', 'home essential',
       'india', 'travel', 'open', 'ott & social', 'jewelery', 'cd',
       'beauty and personal care', 'customer data', 'adv+', 'engaged',
       'designer clothing', 'women innerwear', 'mof + bof',
       'viewed + atc - 180 days', 'sale', 'hotel', 'retarget', 'brands',
       'women', "wedding's", 'retargeting', 'cx data', 'parties',
       'bride & wedding', 'india designers', 'magazines',
       'open targeting', 'wedding', 'women clothing', 'party wear',
       'luxury', "royal's", 'targeting', 'clothing brands', 'lived india',
       'summer wedding', 'lux designers', 'latest fashion', 'fitness',
       'online shoppers', "women's brands", 'vintage clothing',
       "holiday's", 'saree', 'influencers', 'shopify data lookalike'],
      dtype=object)

In [33]:
# same for ad name
Ad_parts = campaigns['Ad Name'].str.split('|', expand=True)

In [34]:
Ad_parts.head(30)

,0,1,2,3,4,5,6,7
0,custom,video 2,eoss,15th dec 25 â copy,None,None,None,None
1,custom,video 2,eoss,15th dec 25 â copy,None,None,None,None
3,custom,video 2,eoss,15th dec 25 â copy,None,None,None,None
4,custom,video 2,eoss,15th dec 25 â copy,None,None,None,None
10,unknown,None,None,None,None,None,None,None
12,custom,catalog,15th dec 25,None,None,None,None,None
17,catalogue,set + dress,,active,instock,None,None,None
18,catalogue,set + dress,,active,instock,None,None,None
19,catalogue,set + dress,,active,instock,None,None,None
20,catalogue,set + dress,,active,instock,None,None,None


In [35]:
campaigns['ad_type'] = ad_parts[0]
campaigns['content_type'] = ad_parts[1]
campaigns['category'] = ad_parts[2]

In [36]:
campaigns[['ad_type','content_type','category']]=campaigns[['ad_type','content_type','category']].fillna('unknown')

In [37]:
campaigns['ad_type'] = campaigns['ad_type'].str.strip().str.lower()
campaigns['content_type'] = campaigns['content_type'].str.strip().str.lower()
campaigns['category'] = campaigns['category'].str.strip().str.lower()

In [38]:
campaigns[['ad_type','content_type','category']]=campaigns[['ad_type','content_type','category']].replace('none','unknown')

In [39]:
campaigns['category'].unique()

array(['sales', 'unknown', 'cac', 'sale', 'sales - women',
       'catalogue only', 'open audence', 'ios', 'women', 'home essential',
       'message - video', 'apparel', 'ott & social',
       'beauty and personal care', 'purchase', 'leads', 'us/ca/aus'],
      dtype=object)

In [40]:
# now our data is cleaned and we have handled nulls also 

In [41]:
# now calculations of CTR, CPM, ROI and CPC
# using apply and conditions so that we do not encounter error while dividing by 0 as we have filled nulls with 0

In [42]:
campaigns['CTR'] = campaigns.apply(
    lambda x: x['Clicks (all)']/x['Impressions'] if x['Impressions'] != 0 else 0,
    axis=1
)

campaigns['CPC'] = campaigns.apply(
    lambda x: x['Amount Spent (INR)']/x['Clicks (all)'] if x['Clicks (all)'] != 0 else 0,
    axis=1
)

campaigns['CPM'] = campaigns.apply(
    lambda x: (x['Amount Spent (INR)']/x['Impressions'])*1000 if x['Impressions'] != 0 else 0,
    axis=1
)

campaigns['ROI'] = campaigns.apply(
    lambda x: x['Purchases Conversion Value (INR)']/x['Amount Spent (INR)'] if x['Amount Spent (INR)'] != 0 else 0,
    axis=1
)

In [43]:
campaigns.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3734 entries, 0 to 10026
Data columns (total 38 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   Data Source name                             3734 non-null   object        
 1   Date                                         3734 non-null   datetime64[ns]
 2   Campaign Name                                3734 non-null   object        
 3   Campaign Effective Status                    3734 non-null   object        
 4   Ad Set Name                                  3734 non-null   object        
 5   Ad Name                                      3734 non-null   object        
 6   Country Funnel                               3734 non-null   object        
 7   Geo Location Segment                         3734 non-null   object        
 8   FB Spent Funnel (INR)                        3734 non-null   float64       
 9   A

In [44]:
# now we do same cleaning for shopify data frame 

In [45]:
Shopify_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5680 entries, 0 to 5679
Data columns (total 38 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Data Source name           5252 non-null   object 
 1   Date                       5242 non-null   object 
 2   Currency                   5258 non-null   object 
 3   Sales Channel              5258 non-null   object 
 4   Transaction Timestamp      5240 non-null   object 
 5   Order Created At           5273 non-null   object 
 6   Order Updated At           5257 non-null   object 
 7   Order ID                   5230 non-null   float64
 8   Order Name                 5257 non-null   object 
 9   Country Funnel             5256 non-null   object 
 10  Geo Location Segment       5269 non-null   object 
 11  Billing Country            5227 non-null   object 
 12  Billing Province           4995 non-null   object 
 13  Billing City               5068 non-null   objec

In [46]:
date_cols = ['Date','Transaction Timestamp','Order Created At','Order Updated At']
Shopify_sales[date_cols] = Shopify_sales[date_cols].apply(pd.to_datetime,errors = 'coerce') # for multiple column

In [47]:
Shopify_sales.duplicated().sum()

np.int64(22)

In [48]:
Shopify_sales.drop_duplicates(inplace = True)

In [49]:
Shopify_sales.isnull().sum()

Data Source name              428
Date                         3430
Currency                      422
Sales Channel                 419
Transaction Timestamp         750
Order Created At              711
Order Updated At              732
Order ID                      449
Order Name                    422
Country Funnel                424
Geo Location Segment          408
Billing Country               452
Billing Province              683
Billing City                  611
Order Tags                   1812
Product ID                   2525
Product Title                2347
Product Tags                 2300
Product Type                 2348
Variant Title                2335
Gross Sales (INR)             458
Net Sales (INR)               454
Total Sales (INR)             451
Orders                        459
Returns (INR)                 474
Return Rate                  2456
Items Sold                    589
Items Returned               5376
Average Order Value (INR)    3402
New Customer O

In [50]:
# here also we have date column null and they are important  for analysis here we will drop rows with  nulls for  date columns 
Shopify_sales = Shopify_sales.dropna(subset=date_cols)

In [51]:
#our Order name column values are  #101 #102
Shopify_sales['Order Name'] = Shopify_sales['Order Name'].str.replace('#','')

In [52]:
# now our duplicates and date column are fixed let us fix null values for another columns as well 
text_cols_correction = ['Data Source name','Currency','Sales Channel','Order Name','Country Funnel','Geo Location Segment',
             'Billing Country','Billing Province','Billing City','Order Tags','Product Title','Product Tags','Product Type','Variant Title',
             'SKU','Customer Sale Type','Shipping Country'
]# we need these column later for string functions

Shopify_sales[text_cols_correction] = Shopify_sales[text_cols_correction].fillna('Unknown')

numeric_cols = ['Order ID','Product ID','Gross Sales (INR)','Net Sales (INR)','Total Sales (INR)','Orders','Returns (INR)',
    'Return Rate','Items Sold','Items Returned','Average Order Value (INR)','New Customer Orders',
    'Returning Customer Orders','Average Items Per Order','Discounts (INR)','Row Count','Customer ID'
]  # in real world problem we do not fill order id with zero but its is assesment so....
Shopify_sales[numeric_cols] = Shopify_sales[numeric_cols].fillna(0)

In [53]:
# okay nulls are handled  now we correct our string value by normalizing them 

In [54]:
for col in text_cols_correction:
    Shopify_sales[col] = Shopify_sales[col].str.strip().str.lower()

In [55]:
# now here also some column have nan as values we need to fill them with unknown
Shopify_sales[text_cols_correction] = Shopify_sales[text_cols_correction].replace('nan','unknown')

In [56]:
Shopify_sales['Country Funnel'] = Shopify_sales['Country Funnel'].str.title()

In [57]:
Shopify_sales['Data Source name'].unique()

array(['brand a', 'unknown', 'brand b'], dtype=object)

In [58]:
Shopify_sales['AOV_calculated'] = Shopify_sales.apply(
    lambda x: x['Total Sales (INR)']/x['Orders'] if x['Orders'] != 0 else 0,
    axis=1
)

In [59]:
# now we have both data cleaned we cna load them into sql but we need to do some more fixes first

In [60]:
# first we need to change column names sql prefer _  instead of spaces 

In [61]:
campaigns.columns = campaigns.columns.str.strip().str.lower().str.replace('[^a-z0-9]+', '_', regex=True)
campaigns.columns = campaigns.columns.str.replace('inr_', 'inr', regex=True)
campaigns.columns = campaigns.columns.str.replace('all_', 'all', regex=True)

In [62]:
Shopify_sales.columns = Shopify_sales.columns.str.strip().str.lower().str.replace('[^a-z0-9]+', '_', regex=True)
Shopify_sales.columns = Shopify_sales.columns.str.strip().str.lower().str.replace('inr_', 'inr', regex=True)

In [63]:
Shopify_sales.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1443 entries, 0 to 5675
Data columns (total 39 columns):
 #   Column                     Non-Null Count  Dtype              
---  ------                     --------------  -----              
 0   data_source_name           1443 non-null   object             
 1   date                       1443 non-null   datetime64[ns]     
 2   currency                   1443 non-null   object             
 3   sales_channel              1443 non-null   object             
 4   transaction_timestamp      1443 non-null   datetime64[ns, UTC]
 5   order_created_at           1443 non-null   datetime64[ns, UTC]
 6   order_updated_at           1443 non-null   datetime64[ns, UTC]
 7   order_id                   1443 non-null   float64            
 8   order_name                 1443 non-null   object             
 9   country_funnel             1443 non-null   object             
 10  geo_location_segment       1443 non-null   object             
 11  billing_c

In [64]:
# now lets us make a data base and laod data in our databse 

In [67]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///growify.db") # creates a sql database file

In [68]:
# we have a sql file where we have written queries for table and other queries here we will just populate them

In [70]:
dim_campaign = campaigns[['data_source_name','campaign_name','channel', 'country_funnel','region',
    'funnel_stage', 'brand', 'persona','category'
]].drop_duplicates().reset_index(drop=True)

dim_campaign['campaign_id'] = dim_campaign.index + 1
cols = ['campaign_id'] + [col for col in dim_campaign.columns if col != 'campaign_id']
dim_campaign = dim_campaign[cols] #reorderd column as per our sql table

In [71]:
dim_campaign.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 385 entries, 0 to 384
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   campaign_id       385 non-null    int64 
 1   data_source_name  385 non-null    object
 2   campaign_name     385 non-null    object
 3   channel           385 non-null    object
 4   country_funnel    385 non-null    object
 5   region            385 non-null    object
 6   funnel_stage      385 non-null    object
 7   brand             385 non-null    object
 8   persona           385 non-null    object
 9   category          385 non-null    object
dtypes: int64(1), object(9)
memory usage: 30.2+ KB


In [72]:
dim_campaign.to_sql(
    name = "dim_campaign",
    con = engine,
    index = False,
    if_exists = "replace")

385

In [70]:
# to create fact table we need data from both table and merge them like we do join in sql

In [73]:
#table 1
campaign_df = campaigns[['date','country_funnel','geo_location_segment','impressions','clicks_all','amount_spent_inr','region',
                          'ctr','cpc','cpm','roi']]
campaign_df['campaign_id'] = campaign_df.index + 1#creating a column campaign id as we have in dim_campaign for joining 
cols = ['campaign_id'] + [col for col in campaign_df.columns if col != 'campaign_id']
campaign_df= campaign_df[cols]
#table 2
shopify_agg = Shopify_sales.groupby(
    ['date', 'country_funnel', 'geo_location_segment']
).agg({
    'total_sales_inr': 'sum',
    'orders': 'sum'
}).reset_index()

shopify_agg.rename(columns={
    'total_sales_inr': 'total_sales',
}, inplace=True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8644\3133123077.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  campaign_df['campaign_id'] = campaign_df.index + 1#creating a column campaign id as we have in dim_campaign for joining


In [74]:
#now our fact_table

In [75]:
fact_table = pd.merge(
    campaign_df,
    shopify_agg,
    on=['date', 'country_funnel', 'geo_location_segment'],
    how='left'   # IMPORTANT
)


In [76]:
fact_table['month'] = fact_table['date'].dt.month

In [80]:
fact_table['amount_spent_inr'] = fact_table['amount_spent_inr'].fillna(0)
fact_table['orders'] = fact_table['orders'].fillna(0)#cleanup

In [81]:
fact_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3734 entries, 0 to 3733
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   campaign_id           3734 non-null   int64         
 1   date                  3734 non-null   datetime64[ns]
 2   country_funnel        3734 non-null   object        
 3   geo_location_segment  3734 non-null   object        
 4   impressions           3734 non-null   float64       
 5   clicks_all            3734 non-null   float64       
 6   amount_spent_inr      3734 non-null   float64       
 7   region                3734 non-null   object        
 8   ctr                   3734 non-null   float64       
 9   cpc                   3734 non-null   float64       
 10  cpm                   3734 non-null   float64       
 11  roi                   3734 non-null   float64       
 12  total_sales           1945 non-null   float64       
 13  orders            

In [82]:
fact_table.to_sql(
    name = "fact_sales",
    con = engine,
    index = False,
    if_exists ="replace")

3734

In [83]:
#date table from both existing tables
date_df = pd.DataFrame()
#date_df['date'] = pd.to_datetime(campaigns['date'].append(shopify['date']).unique())
# it did not wroked when i searched it showed new  pandas version removed append so i use concat
date_series = pd.concat([campaigns['date'], Shopify_sales['date']])

date_df = pd.DataFrame({
    'date': pd.to_datetime(date_series.unique())
})
date_df['year'] = date_df['date'].dt.year
date_df['month'] = date_df['date'].dt.month
date_df['month_name'] = date_df['date'].dt.strftime('%B')
date_df['quarter'] = date_df['date'].dt.quarter
date_df['week'] = date_df['date'].dt.isocalendar().week

In [84]:
date_df.to_sql(
    name = "dim_date",
    con = engine,
    index = False,
    if_exists ="replace")

36

In [85]:
pd.read_sql("SELECT * FROM dim_campaign LIMIT 5",engine)#working 

,campaign_id,data_source_name,campaign_name,channel,country_funnel,region,funnel_stage,brand,persona,category
0,1,brand a,unknown,tof,India,unknown,unknown,unknown,lal 3-5%,sales
1,2,brand a,growify | 11th june | tof | abo | india | lal ...,tof,India,abo,tof,growify,lal 3-5%,sales
2,3,brand a,growify | 11th june | tof | abo | india | lal ...,unknown,Unknown,abo,tof,growify,unknown,unknown
3,4,brand a,growify | 16th june | tof | india | asc | cac ...,tof,India,india,tof,growify,asc,cac
4,5,brand a,growify | 23rd sept | tof | abo | sales,market places - tof - sales - growify,India,abo,tof,growify,unknown,unknown


In [86]:
pd.read_sql("select * from fact_sales limit 5",engine) #checking

,campaign_id,date,country_funnel,geo_location_segment,impressions,clicks_all,amount_spent_inr,region,ctr,cpc,cpm,roi,total_sales,orders,month
0,1,2026-02-01 00:00:00.000000,India,india,0.0,0.0,0.0,unknown,0.0,0.0,0.0,0.0,1287398.0,299.0,2
1,2,2026-03-01 00:00:00.000000,India,india,0.0,0.0,0.0,abo,0.0,0.0,0.0,0.0,5430046.0,448.0,3
2,4,2026-01-01 00:00:00.000000,India,india,0.0,0.0,0.0,abo,0.0,0.0,0.0,0.0,2313352.0,585.0,1
3,5,2026-11-01 00:00:00.000000,Unknown,india,0.0,0.0,0.0,abo,0.0,0.0,0.0,0.0,6470.0,1.0,11
4,11,2026-01-01 00:00:00.000000,India,unknown,0.0,0.0,0.0,india,0.0,0.0,0.0,0.0,150.0,0.0,1


In [87]:
# since our odbc driver is not wprking we are not able to directly load file in power bi by using connection

In [88]:
# so we are going to use basic way of creating file from here

In [89]:
dim_camp_table = pd.read_sql("SELECT * FROM dim_campaign",engine)

In [90]:
fact_sales_table = pd.read_sql("SELECT * FROM fact_sales",engine)

In [91]:
dim_date_table =  pd.read_sql("SELECT * FROM dim_date",engine)

In [92]:
export_path  = r"E:\Growify project"

In [93]:
dim_camp_table.to_csv(f"{export_path}\\dim_camp_table.csv",index = False)

In [94]:
fact_sales_table.to_csv(f"{export_path}\\fact_sales_table.csv",index = False)
dim_date_table.to_csv(f"{export_path}\\dim_date_table.csv",index = False)